# HW1 Assignment: Indigenous Language AI Benchmark — Nupe Track
### Group ID: `group_01`
### Members: Name	Matric Number
Abdullahi Abba Ladan	U22/FNS/CSC/1355
Adua Usman Mohammed	U22/FEA/SED/1079
Ahmad Aliyu	U22/FEA/SED/1294
Aliyu Ahmad	U22/FNS/CSC/1060
Aliyu Salihu Musa	U22/FNS/CSC/1159
Baba Kabiru Alhaji	U22/FEA/SED/1347
Fatima Bello	U22/FEA/SED/1060
Haruna Suleiman	U22/FEA/SED/1197
Hassan Shehu	U22/FEA/SED/1431
Hassan Shehu	U22/FEA/SED/1413
Ibrahim Asmau Umar	U22/FNS/CSC/1153
Ibrahim Hauwa	U22/FNS/CSC/1258
Idris Umar Muhammed	U22/FNS/CSC/1305
Isyaku Ibrahim Makun	U22/FEA/SED/1361
Mohammed Ahmed Liman	U22/FEA/SED/1245
Mohammed Idris	U22/FNS/CSC/1110
Muhammad Ahmad Tijjani	U22/FNS/CSC/1216
Nagenu Muhammed Yusuf	U22/FNS/CSC/1148
Uriah Tswanya (you)	U22/FNS/CSC/1274
Usman Abubakar Sadiq	U22/FNS/CSC/1259
Yahaya Kudu Nagya	U22/FNS/CSC/1301
Yusuf Mohammed Yusuf	U23/FEA/SED/2010
### Language: Nupe (Nupeci)

Implements all 4 required parts per the assignment PDF:
1. Data Collection & Web Scraping (min. 2,500 sentences) — 4 marks
2. Regex Normalization, Custom Tokenization & Stop Words (30+, with translations) — 5 marks
3. Zipf's Law Analysis — 4 marks
4. Unigram + Bigram Language Models, Laplace Smoothing, Perplexity — 5 marks

**Academic integrity:** every URL scraped below must be a real source you access directly with
this notebook's own code. No Hugging Face / Kaggle / pre-tokenized datasets — the autograder
and a blind test set will catch mismatched/copied data.


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import math
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

GROUP_ID = "group_01"   # TODO: set to your actual group id, e.g. "group_03"
LANGUAGE = "nupe"


## Part 1: Data Collection & Web Scraping (4 marks)

Target: **minimum 2,500 sentences** for Nupe. You'll likely need several source pages/documents
to hit this — one blog post won't be enough. Document every source URL, scrape timestamp, and
your collection method (this is a required deliverable, not optional commentary).

**Replace the URL list with your group's own real Nupe-language sources** — news sites, community
blogs, translated religious/literary texts, or other public Nupe text. Do not reuse another
group's exact source list.


In [ ]:
import json, os

GROUP_ID = "group_01"
LANGUAGE = "nupe"

with open("../../data/nupe/raw/raw_data_group_01_merged.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} raw records")

raw_dir = f"../../data/{LANGUAGE}/raw"
os.makedirs(raw_dir, exist_ok=True)
raw_path = f"{raw_dir}/raw_data_{GROUP_ID}.jsonl"

with open(raw_path, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {raw_path}")
print(f"Total sentences: {len(records)} (target: 2,500+)")


In [ ]:
# (Export already handled in the cell above -- this cell intentionally left as a no-op)
print("Raw data already saved in the previous cell.")


## Part 2: Normalization, Custom Tokenization & Stop Words (5 marks)

Required output format for `cleaned_corpus_<group_id>.txt`:
- **Exactly one sentence per line**
- Tokens single-space separated
- **Punctuation detached** from words (e.g. `word .` not `word.`)
- Text **lowercased**, while **strictly preserving diacritics and tone marks** (ẹ, ọ, ṇ, etc.)
- Custom rule-based tokenizer only — no NLTK `word_tokenize`, no spaCy


In [ ]:
def split_sentences(text):
    text = re.sub(r"<[^>]+>", " ", text)              # strip HTML/XML markup
    text = re.sub(r"[\r\n\t\x0b\x0c]+", " ", text)   # strip control chars
    text = re.sub(r"\s+", " ", text).strip()
    raw_sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in raw_sentences if s.strip()]

def tokenize_sentence(sentence):
    raw_tokens = re.findall(r"[^\W\d_]+(?:['\u2019][^\W\d_]+)?|[.!?,;:]", sentence, re.UNICODE)
    tokens = [t.lower() for t in raw_tokens]
    return tokens

all_sentences_tokens = []  # list of token-lists, one per sentence
for rec in records:
    for sent in split_sentences(rec["raw_text"]):
        toks = tokenize_sentence(sent)
        if toks:
            all_sentences_tokens.append(toks)

print(f"Total sentences: {len(all_sentences_tokens)}")
if len(all_sentences_tokens) < 2500:
    print(f"Below the 2,500-sentence target ({len(all_sentences_tokens)} sentences) -- shortfall documented honestly in group_report.md due to Nupe's severe data scarcity.")
else:
    print("2,500+ sentence target met.")


In [ ]:
# Stop word list -- 30+ Nupe function words WITH English translations.
NUPE_STOPWORDS = {
    "a": "he/she/it (subject pronoun)",
    "u": "you (subject pronoun)",
    "wun": "they",
    "eyi": "this",
    "ele": "that",
    "ki": "to/for",
    "gi": "and",
    "be": "in/at",
    "na": "with",
    "o": "or",
    "sa": "but",
    "ba": "not",
    "ke": "of",
    "yi": "this (determiner)",
    "wo": "you (object)",
    "mi": "I / me",
    "de": "to be / is",
    "ga": "shall / will",
    "la": "to take/hold",
    "lo": "to go",
    "ma": "not / did not",
    "e": "it / he-she (short form)",
    "ci": "to be (copula)",
    "da": "to come",
    "fu": "you (possessive)",
    "nya": "of / belonging to",
    "bo": "at / in (locative)",
    "na e": "that / which",
    "shi": "to sit/stay",
    "tsu": "to reach/arrive",
    "kata": "town/place",
    "gan": "to say/do",
}
assert len(NUPE_STOPWORDS) >= 30, "Need at least 30 stop words with translations before submitting."

def remove_stopwords(tokens):
    return [t for t in tokens if t not in NUPE_STOPWORDS]


In [ ]:
# Build cleaned corpus: one sentence per line, punctuation detached, stopwords kept in the
# corpus itself (stopword removal is applied only where a task calls for it, e.g. some Zipf views) —
# check your lecturer's expectation here; by default we keep full sentences for the LM in Part 4
# and provide a stopword-filtered token stream separately for analysis.

processed_dir = f"../../data/{LANGUAGE}/processed"
os.makedirs(processed_dir, exist_ok=True)
processed_path = f"{processed_dir}/cleaned_corpus_{GROUP_ID}.txt"

with open(processed_path, "w", encoding="utf-8") as f:
    for toks in all_sentences_tokens:
        f.write(" ".join(toks) + "\n")

print(f"Saved {processed_path}")

# Flat token stream (stopwords removed) — used for Zipf's Law / vocabulary stats
all_tokens_flat = [t for toks in all_sentences_tokens for t in toks]
all_tokens_no_stop = remove_stopwords(all_tokens_flat)
print(f"Total tokens: {len(all_tokens_flat)} | after stopword removal: {len(all_tokens_no_stop)}")


## Part 3: Zipf's Law Analysis (4 marks)

Compute rank-frequency, fit `log(f) = C - s*log(r)`, and discuss how diacritics/subdot vowels
and orthographic complexity affect vocabulary size and the frequency distribution.


In [ ]:
freqs = Counter(all_tokens_flat)
V = len(freqs)  # unique vocabulary size — needed for the Google Form
sorted_freqs = sorted(freqs.values(), reverse=True)
ranks = np.arange(1, len(sorted_freqs) + 1)

log_ranks = np.log(ranks)
log_freqs = np.log(sorted_freqs)

slope, intercept = np.polyfit(log_ranks, log_freqs, 1)
zipf_exponent = -slope  # since log(f) = C - s*log(r)  =>  slope = -s

print(f"Vocabulary size V = {V}")
print(f"Estimated Zipfian exponent s = {zipf_exponent:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(log_ranks, log_freqs, "o", markersize=3, label="Observed")
plt.plot(log_ranks, slope * log_ranks + intercept, "r-", label=f"Fit (s={zipf_exponent:.2f})")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf's Law — Nupe Corpus")
plt.legend()
plt.savefig("zipf_plot.png", dpi=150, bbox_inches="tight")
plt.show()


**Synthesis (fill in before submitting):** discuss how subdot vowels/tone marks affect
tokenization granularity and vocabulary expansion — e.g. does treating diacritic variants as
distinct tokens inflate V and flatten the frequency curve compared to an orthography without them?


## Part 4: Unigram + Bigram Language Models, Laplace Smoothing, Perplexity (5 marks)

Perplexity formula per the assignment spec (base-2):

`Perplexity(W) = 2^( -(1/N) * Σ log2 P(w_i | w_{i-1}) )`


In [ ]:
class NGramModel:
    def __init__(self):
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.vocab = set()
        self.total_unigrams = 0

    def fit(self, sentences_tokens):
        for toks in sentences_tokens:
            seq = ["<s>"] + toks + ["</s>"]
            self.vocab.update(seq)
            self.unigram_counts.update(seq)
            self.total_unigrams += len(seq)
            for i in range(len(seq) - 1):
                self.bigram_counts[(seq[i], seq[i + 1])] += 1

    def unigram_prob(self, w):
        V = len(self.vocab)
        return (self.unigram_counts[w] + 1) / (self.total_unigrams + V)

    def bigram_prob(self, w1, w2):
        V = len(self.vocab)
        return (self.bigram_counts[(w1, w2)] + 1) / (self.unigram_counts[w1] + V)

    def perplexity_bigram(self, test_sentences_tokens):
        log2_prob_sum = 0.0
        N = 0
        for toks in test_sentences_tokens:
            seq = ["<s>"] + toks + ["</s>"]
            for i in range(len(seq) - 1):
                p = self.bigram_prob(seq[i], seq[i + 1])
                log2_prob_sum += math.log2(p)
                N += 1
        return 2 ** (-log2_prob_sum / N)

model = NGramModel()
model.fit(all_sentences_tokens)
print(f"Vocabulary size (incl. <s>/</s>): {len(model.vocab)}")
print(f"Unique bigrams: {len(model.bigram_counts)}")


In [ ]:
# Evaluate on the instructor-provided blind test set: tests/test_nupe_unseen.txt
# (one sentence per line, same tokenization rules applied)
with open("../../tests/test_nupe_unseen.txt", encoding="utf-8") as f:
    test_lines = [line.strip() for line in f if line.strip()]

test_sentences_tokens = [tokenize_sentence(line) for line in test_lines]
perplexity = model.perplexity_bigram(test_sentences_tokens)
print(f"Bigram model perplexity on blind test set: {perplexity:.4f}")

# This exact number goes into the Google Form.


## Final Metrics Summary — fill in before submitting

- Total raw sentence count: `<fill in>`
- Unique vocabulary size (V): `<fill in>`
- Zipfian exponent s: `<fill in>`
- Bigram perplexity on blind test set: `<fill in>`

Copy these into `group_report.md` and the Google Form.
